In [8]:
from torchvision import datasets, transforms
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from google.colab import drive
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

transform = transforms.Compose([transforms.ToTensor(),
                             transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
train_data = datasets.CIFAR10(root='/content/drive/MyDrive/cifar_data', train=True, download=False, transform=transform)
test_data = datasets.CIFAR10(root='/content/drive/MyDrive/cifar_data', train=False, download=False, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [7]:
class CIFAR10(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3)
    self.bn1 = nn.BatchNorm2d(32)

    self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
    self.bn2 = nn.BatchNorm2d(64)
    self.pool = nn.MaxPool2d(2, 2)

    self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)
    self.bn3 = nn.BatchNorm2d(128)

    self.conv4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3)
    self.bn4 = nn.BatchNorm2d(128)


    self.fc1 = nn.Linear(128,256)
    self.dropout = nn.Dropout(0.5)
    self.fc2 = nn.Linear(256, 10)

  def forward(self, x):
      x = self.pool(F.relu(self.bn1(self.conv1(x))))
      #print("After conv1 n pool:", x.shape)

      x = self.pool(F.relu(self.bn2(self.conv2(x))))
      #print("After conv2 n pool:", x.shape)

      x = F.relu(self.bn3(self.conv3(x)))
      #print("After conv3:", x.shape)

      x = F.relu(self.bn4(self.conv4(x)))
      #print("After conv4:", x.shape)

      x = self.pool(x)
      #print("After final pool:", x.shape)

      x = x.view(x.size(0), -1)
      #print("Flatten size:", x.shape)

      x = F.relu(self.fc1(x))
      x = self.dropout(x)
      x = self.fc2(x)
      return x

model = CIFAR10()
#dummy = torch.randn(1, 3, 32, 32)
#out = model(dummy)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(15):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()


    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg loss: {avg_loss:.4f}")

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = correct / total
    print(f"Validation accuracy: {acc:.3f}")


Epoch 1 | Avg loss: 1.1613
Validation accuracy: 0.663
Epoch 2 | Avg loss: 0.8136
Validation accuracy: 0.711
Epoch 3 | Avg loss: 0.6729
Validation accuracy: 0.728
Epoch 4 | Avg loss: 0.5700
Validation accuracy: 0.738
Epoch 5 | Avg loss: 0.4873
Validation accuracy: 0.758
Epoch 6 | Avg loss: 0.4110
Validation accuracy: 0.745
Epoch 7 | Avg loss: 0.3459
Validation accuracy: 0.769
Epoch 8 | Avg loss: 0.2937
Validation accuracy: 0.775
Epoch 9 | Avg loss: 0.2397
Validation accuracy: 0.772
Epoch 10 | Avg loss: 0.2149
Validation accuracy: 0.771
Epoch 11 | Avg loss: 0.1778
Validation accuracy: 0.755
Epoch 12 | Avg loss: 0.1627
Validation accuracy: 0.769
Epoch 13 | Avg loss: 0.1385
Validation accuracy: 0.769
Epoch 14 | Avg loss: 0.1289
Validation accuracy: 0.759
Epoch 15 | Avg loss: 0.1189
Validation accuracy: 0.750
